# HybridDataset walkthrough

Minimal, valid notebook showing HybridDataset mixed routing and source overrides.

In [1]:
from __future__ import annotations

import datetime as dt
from pathlib import Path
from tempfile import TemporaryDirectory

from sqlalchemy import Date, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

from boti_data import DataHelper, HybridDataset


class Base(DeclarativeBase):
    pass


class HistoricalEvent(Base):
    __tablename__ = "historical_events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))


class LiveEvent(Base):
    __tablename__ = "live_events"

    id: Mapped[int] = mapped_column(Integer, primary_key=True)
    event_date: Mapped[dt.date] = mapped_column(Date())
    status: Mapped[str] = mapped_column(String(16))


In [2]:
tmp_dir = TemporaryDirectory()
db_path = Path(tmp_dir.name) / "hybrid_notebook.db"
engine = create_engine(f"sqlite:///{db_path}")

Base.metadata.create_all(engine)
with Session(engine) as session:
    session.add_all([
        HistoricalEvent(id=1, event_date=dt.date(2026, 4, 14), status="hist"),
        HistoricalEvent(id=2, event_date=dt.date(2026, 4, 16), status="hist"),
        LiveEvent(id=10, event_date=dt.date(2026, 4, 18), status="live"),
        LiveEvent(id=11, event_date=dt.date(2026, 4, 19), status="live"),
    ])
    session.commit()


In [3]:
historical = DataHelper(
    backend="sqlalchemy",
    connection_url=f"sqlite:///{db_path}",
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="historical_events",
)
live = DataHelper(
    backend="sqlalchemy",
    connection_url=f"sqlite:///{db_path}",
    poolclass="sqlalchemy.pool.NullPool",
    query_only=False,
    table="live_events",
)
dataset = HybridDataset(historical, live, date_field="event_date", split_date="2026-04-18")


In [4]:
mixed = dataset.load(start="2026-04-14", end="2026-04-19", return_type="auto")
historical_df = dataset.pandas.load(start="2026-04-14", end="2026-04-17", source="historical")
live_df = dataset.pandas.load(start="2026-04-18", end="2026-04-19", source="live")
async_df = await dataset.aload(start="2026-04-16", end="2026-04-18", return_type="pandas")

mixed_rows = mixed.shape[0]
if hasattr(mixed_rows, "compute"):
    mixed_rows = mixed_rows.compute()

summary = {
    "mixed_type": type(mixed).__name__,
    "mixed_rows": int(mixed_rows),
    "historical_rows": len(historical_df),
    "live_rows": len(live_df),
    "async_rows": len(async_df),
}
summary


/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: Distributed SQL task payloads will carry the raw DSN credential because 'worker_connection_env_var' is not set on SqlDatabaseConfig. Set 'worker_connection_env_var' to the name of an environment variable that resolves the DSN on each worker to avoid serializing credentials.
  self._worker_config = WorkerSqlConfig.from_database_config(config)
WorkerSqlConfig created with raw DSN fallback; set worker_connection_env_var to avoid credential serialization.
/Users/lvalverdeb/TeamDev/repo-split/boti-data/src/boti_data/db/partitioned_loader.py:36: UserWarning: Distributed SQL task payloads will carry the raw DSN credential because 'worker_connection_env_var' is not set on SqlDatabaseConfig. Set 'worker_connection_env_var' to the name of an environment variable that resolves the DSN on each worker to avoid serializing credentials.
  self._worker_config = WorkerSqlConfig.from_database_config(co

{'mixed_type': 'DataFrame',
 'mixed_rows': 4,
 'historical_rows': 2,
 'live_rows': 2,
 'async_rows': 2}

In [5]:
dataset.close()
engine.dispose()
tmp_dir.cleanup()
